# BirdCLEF+ 2025 - Pretrained models: noisy-student round 1, student training (TEACHER = main)

**T4 GPU, internet off, no submission.** This follows the pre-registered spec (STATE.md, "Noisy student PRE-REGISTRATION" and phase "noisy-student round 1"). The two student notebooks (`ns-student-main`, `ns-student-ablation`) are identical except for the `TEACHER` line.

**Init:** the bird-pretrained `tf_efficientnetv2_s_in21k` backbone `models_2025/tf_efficientnetv2_s_in21k_pretrain_from_bigXCV2Ext_swa.ckpt`, loaded exactly as in `a1-v2s-train` (never the teacher weights). drop_path_rate 0.15.

**Focal part (A1/v2 recipe verbatim):** the baseline mel cache, random 5 s crop, the baseline's soundscape-noise p=0.5 at -20..-3 dB, secondary labels 0.5, `zscore_minmax` input, split seed 42.

**Soundscape part:** each epoch uses 2,500 whole `train_soundscapes` files (rng seed 42 + epoch) x 12 windows = 30,000 windows, with the teacher's soft labels from the `ns-pseudo-labels` v1 kernel output (pinned by sha256). The files are decoded once per epoch by 4 DataLoader workers, and the mel is the verified torch mel (`ns-mel-check`, NEAR_IDENTICAL) on the GPU. Windows go into a GPU pool of about 64 files and are drawn at random from it, so a batch mixes windows of many files.

**Batches:** one epoch = one pass over the focal train clips plus the 30,000 windows. The two row types are shuffled together (random positions over the epoch, batch 64, last partial batch dropped). Mixup p=0.5 Beta(0.4,0.4) is applied over the combined batch, with target = elementwise max (v2). Loss = soft-target BCE with a per-row weight of 1.0 focal / 0.5 soundscape; a mixed row gets the lambda-weighted mean of its two weights, and the batch loss is the weighted mean.

**Optimiser and validation:** AdamW 1e-3 one-cycle (pct_start 0.1) over all steps, 15 epochs, AMP as in A1. The clean val AUC (first 5 s, primary label, baseline protocol) is recorded each epoch. `model.pth` = best-by-val as in A1 train (a recorded metric, not used to choose between students), and `model_final.pth` = last epoch.

**Output:** `model.pth`, `model_final.pth`, `cfg.json`, `species.json`, `history.csv`, `student_meta.json`.

In [ ]:
import os, time
import torch

T_NOTEBOOK = time.time()
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
TEACHER = "main"      # "main" | "ablation": the only line that differs between the two student notebooks
assert TEACHER in ("main", "ablation"), TEACHER
PSEUDO_FILE = {"main": "pseudo_main.npy", "ablation": "pseudo_ablation.npy"}[TEACHER]
PSEUDO_KERNEL, PSEUDO_VERSION = "sc4000-birdclef-m3-ns-pseudo-labels", 1
PINNED_SHA256 = {
    "pseudo_main.npy": "0e4fbd0d56ae9456804f32adaef5d80b77666d4629beae999ce4550d0fe8e3f9",
    "pseudo_ablation.npy": "63ddec4fc870b1c7ad6562f25b487ffba309071a3bd193076d70fb572ce69779",
    "soundscape_all_windows.csv": "43d1fbc5cafdda81e01add98f27d4549b1014559bc592155f442a81ebb14c04a",
    "species.json": "6c7d634689e19515c905dbcfd214abdac1e81943c6f95601846b731c2fb60924",   # = a1-v2s-train species.json
}
# On Kaggle every check is on. M3_STRICT only re-enables named checks inside a local smoke test.
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE or name in STRICT

# = the cfg.json of a1-v2s-train (EXPERIMENT v2, A1_BACKBONE v2s_bird, A1_NORM zscore_minmax), plus drop_path_rate
CFG = dict(
    sr=32_000, duration=5, n_fft=2048, hop_length=512, n_mels=128, fmin=50, fmax=14_000,
    model="tf_efficientnetv2_s_in21k", epochs=15, batch_size=64, lr=1e-3, weight_decay=1e-2,
    val_frac=0.2, seed=42,
    random_crop=True, noise=True, noise_prob=0.5, noise_db=(-20, -3),
    mixup=True, mixup_prob=0.5, mixup_alpha=0.4,
    secondary=True, secondary_weight=0.5, focal=False, focal_gamma=2.0, experiment="v2",
    backbone_ckpt="models_2025/tf_efficientnetv2_s_in21k_pretrain_from_bigXCV2Ext_swa.ckpt",
    backbone="v2s_bird", norm="zscore_minmax", norm_top_db=80.0,
    drop_path_rate=0.15,
)
CKPT_WEIGHTS_ONLY = True     # as A1: refuse to unpickle arbitrary objects from the third-party checkpoint
NS = dict(
    teacher=TEACHER, pseudo_file=PSEUDO_FILE,
    files_per_epoch=int(os.environ.get("M3_FILES_PER_EPOCH", "2500")),     # x 12 windows = 30,000 per epoch
    windows_per_file=12,
    sampling="whole files, rng = np.random.default_rng(seed + epoch), without replacement within an epoch",
    focal_weight=1.0, sound_weight=0.5,
    mixed_row_weight="lambda-weighted mean: w = lam * w_i + (1 - lam) * w_perm(i)",
    loss="soft-target BCEWithLogits, mean over classes per row, then sum(w * row_loss) / sum(w) over the batch",
    pool_files=int(os.environ.get("M3_POOL_FILES", "64")),               # GPU shuffle pool, in files
    sound_workers=int(os.environ.get("M3_SOUND_WORKERS", "4")),
    noise_aug="focal rows only (M1 soundscape noise, as A1); soundscape rows are not augmented apart from mixup",
)
if SMOKE:
    CFG["batch_size"] = int(os.environ.get("M3_BATCH", CFG["batch_size"]))
BUDGET_H = float(os.environ.get("M3_BUDGET_H", "1.9"))
GUARD_STEPS = int(os.environ.get("M3_GUARD_STEPS", "50"))
print("TEACHER:", TEACHER, "| pseudo-labels:", PSEUDO_FILE, "| NS:", {k: NS[k] for k in ("files_per_epoch", "pool_files", "sound_workers")})

In [ ]:
import ast, glob, hashlib, importlib, inspect, json, random, sys, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import librosa
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG["seed"])
torch.backends.cuda.matmul.allow_tf32 = False      # as ns-mel-check (T4 has no TF32 anyway)
torch.backends.cudnn.allow_tf32 = False

def find_dir(marker, fallback):
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

def find_in_kernel(name, kernel, env):
    """Folder of `name` inside the output of one attached kernel."""
    if os.environ.get(env):       # local tests only
        d = Path(os.environ[env])
        assert (d / name).exists(), f"{name} not in {d}"
        return d
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one {name} under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

DATA = Path(os.environ["M1_DATA_DIR"]) if os.environ.get("M1_DATA_DIR") else \
    find_dir("birdclef-2025/sample_submission.csv", "../../data")
MELS = Path(os.environ["M1_MELS_DIR"]) if os.environ.get("M1_MELS_DIR") else \
    find_dir("train_mels.npy", "../../outputs/jamie_make_mels")
PL_DIR = find_in_kernel("pseudo_labels_meta.json", PSEUDO_KERNEL, "M3_PL_DIR")
OUT = Path(os.environ.get("M3_OUT_DIR", "."))
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu (smoke test)"
LIBROSA_PAD_MODE = inspect.signature(librosa.feature.melspectrogram).parameters["pad_mode"].default
VERSIONS = dict(torch=torch.__version__, torchaudio=torchaudio.__version__, timm=timm.__version__, librosa=librosa.__version__)
print("versions:", VERSIONS, "| librosa pad_mode:", LIBROSA_PAD_MODE)
print("Data:", DATA, "| mels:", MELS, "| pseudo-labels:", PL_DIR, "| device:", DEVICE_NAME, "| CPUs:", os.cpu_count())

## Pseudo-labels: ns-pseudo-labels v1, pinned by sha256

In [ ]:
t0 = time.time()
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

SHA_FOUND = {name: sha256_of(PL_DIR / name) for name in PINNED_SHA256}
bad = {n: SHA_FOUND[n] for n in PINNED_SHA256 if SHA_FOUND[n] != PINNED_SHA256[n]}
if bad and strict("sha"):
    raise RuntimeError(f"pseudo-label files do not match the pinned ns-pseudo-labels v{PSEUDO_VERSION}: {bad}")
if bad:
    print("SMOKE: sha256 mismatch ignored (synthetic files):", list(bad))
else:
    print(f"PINNED: {PSEUDO_KERNEL} v{PSEUDO_VERSION} | all 4 sha256 match | using {PSEUDO_FILE} ({SHA_FOUND[PSEUDO_FILE][:12]}...)")

species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
pl_species = json.load(open(PL_DIR / "species.json"))
assert pl_species == species, "pseudo-label species order differs from sample_submission / A1"
C = len(species)

windows = pd.read_csv(PL_DIR / "soundscape_all_windows.csv")
assert list(windows.columns) == ["filename", "win"], list(windows.columns)
PSEUDO = np.load(PL_DIR / PSEUDO_FILE)
assert PSEUDO.dtype == np.float32 and PSEUDO.shape == (len(windows), C), (PSEUDO.dtype, PSEUDO.shape)
if not (np.isfinite(PSEUDO).all() and PSEUDO.min() >= 0 and PSEUDO.max() <= 1):
    raise RuntimeError("pseudo-labels non-finite or outside [0, 1]")
if strict("n") and len(windows) != 116712:
    raise RuntimeError(f"expected 116712 windows, found {len(windows)}")

# File table: csv rows of each file, in window order (row i of PSEUDO = row i of the csv)
WPF = NS["windows_per_file"]
FILE_NAMES, FILE_ROWS = [], []
for name, grp in windows.groupby("filename", sort=False):
    w = grp["win"].values
    if not np.array_equal(w, np.arange(len(w))) or len(w) != WPF:
        raise RuntimeError(f"{name}: windows {w.tolist()[:15]} are not 0..{WPF - 1}")
    FILE_NAMES.append(str(name)); FILE_ROWS.append(grp.index.values.astype(np.int64))
SOUND_DIR = DATA / "train_soundscapes"
present = [(SOUND_DIR / n).exists() for n in FILE_NAMES]
if not all(present):
    if strict("files"):
        miss = [n for n, p in zip(FILE_NAMES, present) if not p]
        raise FileNotFoundError(f"{len(miss)} of {len(FILE_NAMES)} soundscapes missing, e.g. {miss[:3]}")
    keep = [i for i, p in enumerate(present) if p]
    FILE_NAMES = [FILE_NAMES[i] for i in keep]; FILE_ROWS = [FILE_ROWS[i] for i in keep]
    print(f"SMOKE: using the {len(FILE_NAMES)} soundscapes present locally")
if strict("n") and len(FILE_NAMES) != 9726:
    raise RuntimeError(f"expected 9726 soundscape files, found {len(FILE_NAMES)}")
if NS["files_per_epoch"] > len(FILE_NAMES):
    raise RuntimeError(f"files_per_epoch {NS['files_per_epoch']} > {len(FILE_NAMES)} files")
nz = float((PSEUDO > 0).mean()); ge = int((PSEUDO.max(axis=1) >= 0.5).sum())
print(f"{len(windows)} windows in {len(FILE_NAMES)} files used | {PSEUDO_FILE}: {100 * nz:.2f}% non-zero, "
      f"{ge} windows with a label >= 0.5, mean {PSEUDO.mean():.5f} | {time.time() - t0:.1f}s", flush=True)

## Focal clips: baseline spectrogram cache, labels and split (as a1-v2s-train)

In [ ]:
t0 = time.time()
X = np.load(MELS / "train_mels.npy")                 # (total_frames, 128) uint8
index = pd.read_csv(MELS / "train_index.csv")
mcfg = json.load(open(MELS / "mels_cfg.json"))
assert "store_db" in mcfg, "old make_mels output (80 dB); rerun make_mels version 2"
for k in ("sr", "n_fft", "hop_length", "n_mels", "fmin", "fmax"):
    assert mcfg[k] == CFG[k], f"mel setting {k} differs from make_mels"
NOISE = np.load(MELS / "noise_mels.npy") if CFG["noise"] else None
W = mcfg["frames_per_5s"]                            # 313 frames = 5 s
print(f"mels {X.shape} {X.nbytes / 1e9:.1f} GB | {len(index)} clips | "
      f"noise {None if NOISE is None else NOISE.shape} | {time.time() - t0:.0f}s")

sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]
print(len(index), "clips,", len(species), "species |",
      f"{np.mean([len(s) > 0 for s in sec_idx]):.1%} clips have usable secondary labels")

idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
print("train", len(idx_tr), "| val", len(idx_va))

In [ ]:
TOP_DB = float(CFG["norm_top_db"])  # model input keeps 80 dB, like the baseline
STORE_DB = float(mcfg["store_db"])  # make_mels stored 120 dB below each clip's peak

def to_db(u8):
    """uint8 from make_mels -> dB relative to the clip's loudest point."""
    return u8.T.astype(np.float32) / 255.0 * STORE_DB - STORE_DB

def to_input(db):
    """Shared with the A1 notebooks, keep all copies identical."""
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

START, NFRAMES = index["start"].values, index["n_frames"].values

class MelDataset(torch.utils.data.Dataset):
    """a1-v2s-train's dataset, unchanged."""
    def __init__(self, idx, train):
        self.idx, self.train = idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        s, n = START[j], NFRAMES[j]
        off = np.random.randint(0, n - W + 1) if (self.train and CFG["random_crop"]) else 0
        db = to_db(X[s + off : s + off + W])                 # (128, W)

        if self.train and CFG["noise"] and np.random.rand() < CFG["noise_prob"]:
            k = np.random.randint(len(NOISE))
            o = np.random.randint(0, NOISE.shape[1] - W + 1)
            nz = to_db(NOISE[k, o : o + W])
            gain_db = np.random.uniform(*CFG["noise_db"])
            # add the two sounds on the power scale, noise peak gain_db below the clip peak
            p = 10.0 ** ((db - db.max()) / 10.0) + 10.0 ** ((nz - nz.max() + gain_db) / 10.0)
            db = 10.0 * np.log10(p)
        x = to_input(db)

        target = np.zeros(len(species), dtype=np.float32)
        if self.train and CFG["secondary"]:
            target[sec_idx[j]] = CFG["secondary_weight"]
        target[y_idx[j]] = 1.0          # validation always uses the primary label only
        return torch.from_numpy(x)[None], torch.from_numpy(target)

class PlanSampler(torch.utils.data.Sampler):
    """Yields this epoch's focal batches (variable size); set .batches before each epoch. Runs in the main process."""
    def __init__(self):
        self.batches = []
    def __iter__(self):
        return iter(self.batches)
    def __len__(self):
        return len(self.batches)

NW = int(os.environ.get("M1_WORKERS", min(4, os.cpu_count())))   # 0 for a local test
PIN = torch.cuda.is_available()
focal_sampler = PlanSampler()
dl_tr = torch.utils.data.DataLoader(MelDataset(idx_tr, True), batch_sampler=focal_sampler, num_workers=NW,
                                    pin_memory=PIN, persistent_workers=NW > 0)
dl_va = torch.utils.data.DataLoader(MelDataset(idx_va, False), batch_size=CFG["batch_size"] * 2,
                                    shuffle=False, num_workers=NW, pin_memory=PIN)

## Soundscape windows: file decode in DataLoader workers, torch mel on the GPU, shuffle pool
We copied `TorchA1Mel` verbatim from `ns-mel-check` (NEAR_IDENTICAL to A1's librosa path: input max abs diff 1.3e-4). It runs outside autocast, in float32.

In [ ]:
# Worker code lives in a module so that the DataLoader workers can import it with fork or spawn.
LIB_SRC = r"""
import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly

class SoundFiles(torch.utils.data.Dataset):
    # One item = one whole soundscape -> ((n_win, sr * duration) float32 windows, position in the list)
    def __init__(self, paths, sr, duration):
        self.paths, self.sr, self.win = [str(p) for p in paths], sr, sr * duration
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        # = the decode of ns-mel-check / ns-a1-all (mono, resample if needed, end padded to whole windows)
        y, sr = sf.read(self.paths[i], dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != self.sr:
            y = resample_poly(y, self.sr, sr).astype(np.float32)
        if len(y) == 0 or not np.isfinite(y).all():
            raise ValueError(f"empty or non-finite audio: {self.paths[i]}")
        n_win = max(1, int(np.ceil(len(y) / self.win)))
        y = np.pad(y, (0, n_win * self.win - len(y)))
        return torch.from_numpy(y.reshape(n_win, self.win)), i
"""
LIB_DIR = Path(tempfile.mkdtemp(prefix="ns_student_lib_"))     # outside the output folder
(LIB_DIR / "ns_sound_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(LIB_DIR))
importlib.invalidate_caches()
import ns_sound_lib as sound_lib


class TorchA1Mel(torch.nn.Module):
    """(B, sr * duration) float32 waveforms -> (B, n_mels, frames) A1 model inputs, on any device.

    Reproduces librosa.feature.melspectrogram(center=True, power=2, Slaney mel, periodic Hann)
    -> librosa.power_to_db(ref=np.max, amin=1e-10, top_db) -> A1's to_input, all per window.
    pad_mode must be the installed librosa's default ("constant" for librosa >= 0.10, "reflect" before).
    """

    def __init__(self, cfg, pad_mode):
        super().__init__()
        self.top_db = float(cfg["norm_top_db"])
        self.norm = cfg["norm"]
        self.amin = 1e-10
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg["sr"], n_fft=cfg["n_fft"], win_length=cfg["n_fft"], hop_length=cfg["hop_length"],
            f_min=float(cfg["fmin"]), f_max=float(cfg["fmax"]), n_mels=cfg["n_mels"],
            window_fn=torch.hann_window,         # periodic=True by default, like scipy get_window("hann", fftbins=True)
            power=2.0, normalized=False, center=True, pad_mode=pad_mode, onesided=True,
            norm="slaney", mel_scale="slaney")

    def db(self, wav):
        """power_to_db(S, ref=np.max, amin=1e-10, top_db) per window."""
        s = self.mel(wav.float())
        dims = (1, 2)
        log_spec = 10.0 * torch.log10(torch.clamp(s, min=self.amin))
        log_spec = log_spec - 10.0 * torch.log10(torch.clamp(s.amax(dim=dims, keepdim=True), min=self.amin))
        return torch.maximum(log_spec, log_spec.amax(dim=dims, keepdim=True) - self.top_db)

    def to_input(self, db):
        """A1's to_input per window (np.std is the population std, hence correction=0)."""
        dims = (1, 2)
        db = torch.clamp(db - db.amax(dim=dims, keepdim=True), min=-self.top_db)
        if self.norm == "zscore_minmax":
            x = (db - db.mean(dim=dims, keepdim=True)) / (db.std(dim=dims, keepdim=True, correction=0) + 1e-6)
            lo, hi = x.amin(dim=dims, keepdim=True), x.amax(dim=dims, keepdim=True)
            return (x - lo) / (hi - lo + 1e-6)
        return (db + self.top_db) / self.top_db

    def forward(self, wav):
        return self.to_input(self.db(wav))


class SoundPool:
    """Fixed-size GPU buffer of soundscape model inputs and soft labels; rows are drawn at random without replacement."""
    def __init__(self, cap, n_mels, frames, n_classes, device):
        self.x = torch.empty((cap, n_mels, frames), dtype=torch.float32, device=device)
        self.y = torch.empty((cap, n_classes), dtype=torch.float32, device=device)
        self.used = np.zeros(cap, bool); self.device = device
    @property
    def count(self):
        return int(self.used.sum())
    def reset(self):
        self.used[:] = False
    def add(self, x, y):
        free = np.flatnonzero(~self.used)[:len(x)]
        if len(free) < len(x):
            raise RuntimeError("sound pool overflow")
        idx = torch.from_numpy(free).to(self.device)
        self.x[idx] = x; self.y[idx] = y; self.used[free] = True
    def take(self, k, rng):
        occ = np.flatnonzero(self.used)
        if k > len(occ):
            raise RuntimeError(f"sound pool has {len(occ)} rows, {k} requested")
        pick = np.sort(rng.choice(occ, size=k, replace=False))
        idx = torch.from_numpy(pick).to(self.device)
        out = self.x[idx], self.y[idx]           # advanced indexing copies
        self.used[pick] = False
        return out


WIN_SAMPLES = CFG["sr"] * CFG["duration"]
mel_gpu = TorchA1Mel(CFG, LIBROSA_PAD_MODE).to(DEVICE)
with torch.no_grad():
    FRAMES = mel_gpu(torch.zeros(1, WIN_SAMPLES, device=DEVICE)).shape[-1]
assert FRAMES == W, f"torch mel gives {FRAMES} frames, M1 cache windows have {W}"
SOUND_PATHS = [SOUND_DIR / n for n in FILE_NAMES]
PSEUDO_T = torch.from_numpy(PSEUDO).to(DEVICE)
POOL_TARGET = NS["pool_files"] * WPF
pool = SoundPool(POOL_TARGET + WPF + CFG["batch_size"], CFG["n_mels"], FRAMES, C, DEVICE)
print(f"sound pool: target {POOL_TARGET} windows, capacity {pool.x.shape[0]} | frames {FRAMES} | sound workers {NS['sound_workers']}")

## Model (a1-v2s-train's backbone loader, unchanged) and the epoch plan

In [ ]:
def find_ckpt(rel):
    """Locate the backbone checkpoint in the attached dataset (M3_CKPT overrides, for local tests)."""
    if os.environ.get("M3_CKPT"):
        return Path(os.environ["M3_CKPT"])
    hits = glob.glob(f"/kaggle/input/**/{rel}", recursive=True) \
        or glob.glob(f"/kaggle/input/**/{Path(rel).name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"backbone checkpoint not found: {rel}")
    return Path(hits[0])

def load_bird_backbone(model, path, min_frac=0.9, show=20):
    """Load a backbone-only checkpoint into a timm model. Key names are inspected at runtime."""
    ckpt = torch.load(path, map_location="cpu", weights_only=CKPT_WEIGHTS_ONLY)
    sd = ckpt
    if isinstance(ckpt, dict):
        print("checkpoint top-level keys:", list(ckpt)[:show])
        for k in ("state_dict", "model_state_dict", "model"):
            if isinstance(ckpt.get(k), dict):
                print("using nested dict:", k); sd = ckpt[k]; break
    sd = {k: v for k, v in sd.items() if torch.is_tensor(v)}
    print(len(sd), "tensors, e.g.", list(sd)[:5])

    target = model.state_dict()
    backbone = {k for k in target if not k.startswith("classifier.")}

    # choose the prefix under which the most keys match the timm model
    def strip(prefix):
        return {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}
    def score(prefix):
        return sum(k in backbone and v.shape == target[k].shape for k, v in strip(prefix).items())
    prefixes = ["model.backbone.", "backbone.", "module.model.backbone.", "module.backbone.",
                "model.", "module.", ""]
    prefix = max(prefixes, key=score)
    print(f"key prefix: {prefix!r} ({score(prefix)} matching keys)")
    stripped = strip(prefix)

    # 1-channel input: same as M1 (in_chans=1 in training and inference), so no conversion
    stem = stripped.get("conv_stem.weight")
    if stem is None:
        raise RuntimeError(f"no conv_stem.weight under prefix {prefix!r}; wrong checkpoint layout?")
    if stem.shape != target["conv_stem.weight"].shape:
        raise RuntimeError(f"conv_stem shape {tuple(stem.shape)} does not fit the model "
                           f"{tuple(target['conv_stem.weight'].shape)} (expected 1-channel input)")

    head = [k for k in stripped if k.startswith("classifier.")]
    wrong = [k for k, v in stripped.items() if k in backbone and v.shape != target[k].shape]
    good = {k: v for k, v in stripped.items() if k in backbone and v.shape == target[k].shape}
    unexpected = [k for k in sd if not (k.startswith(prefix) and k[len(prefix):] in good)]
    res = model.load_state_dict(good, strict=False)
    missing = [k for k in res.missing_keys if not k.startswith("classifier.")]
    frac = len(good) / len(backbone)
    print(f"loaded {len(good)}/{len(backbone)} backbone keys ({frac:.1%}) | head keys dropped "
          f"{len(head)} | shape mismatch {len(wrong)} | missing {len(missing)} | "
          f"unexpected {len(unexpected)}")
    print("missing:", missing[:show])
    print("unexpected:", unexpected[:show])
    print("wrong shape:", wrong[:show])
    if frac < min_frac:
        raise RuntimeError(f"only {frac:.1%} of backbone keys loaded (need {min_frac:.0%})")
    return dict(prefix=prefix, loaded=len(good), backbone_keys=len(backbone), missing=len(missing), unexpected=len(unexpected))

CKPT_PATH = find_ckpt(CFG["backbone_ckpt"])
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species),
                          drop_path_rate=CFG["drop_path_rate"])
LOAD_INFO = load_bird_backbone(model, CKPT_PATH)
drop_probs = [m.drop_prob for m in model.modules() if type(m).__name__ == "DropPath"]
print(f"DropPath modules: {len(drop_probs)}, max drop_prob {max(drop_probs) if drop_probs else None}")
assert drop_probs and max(drop_probs) > 0, "drop_path_rate was not applied"
model = model.to(DEVICE)

EPOCHS = int(os.environ.get("M1_EPOCHS", CFG["epochs"]))
MAX_STEPS = int(os.environ.get("M1_MAX_STEPS", "0"))    # >0 only for a quick local test
BS = CFG["batch_size"]
N_F = len(idx_tr)
N_S = NS["files_per_epoch"] * WPF
STEPS = (N_F + N_S) // BS                                # last partial batch dropped, as A1 (drop_last)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["lr"], total_steps=EPOCHS * STEPS, pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
print(f"epoch = {N_F} focal + {N_S} soundscape rows = {STEPS} steps of {BS} | one-cycle total steps {EPOCHS * STEPS}")

def make_plan(epoch):
    """Files, per-step row counts and focal batches for one epoch, all from rng(seed + epoch)."""
    rng = np.random.default_rng(CFG["seed"] + epoch)
    files = rng.choice(len(FILE_NAMES), size=NS["files_per_epoch"], replace=False)
    total = N_F + N_S
    is_sound = np.zeros(total, bool)
    is_sound[rng.choice(total, size=N_S, replace=False)] = True      # row types shuffled together
    k_s = is_sound[:STEPS * BS].reshape(STEPS, BS).sum(axis=1)
    k_f = BS - k_s
    order = rng.permutation(N_F)                                     # positions in MelDataset(idx_tr)
    bounds = np.concatenate([[0], np.cumsum(k_f)])
    batches = [order[bounds[i]:bounds[i + 1]].tolist() for i in range(STEPS) if k_f[i] > 0]
    return files, k_s, k_f, batches, rng

def mixup(xb, yb, wb):
    """v2 mixup over the combined batch; the row weight is mixed with the same lambda."""
    lam = float(np.random.beta(CFG["mixup_alpha"], CFG["mixup_alpha"]))
    perm = torch.randperm(xb.size(0), device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], torch.maximum(yb, yb[perm]), lam * wb + (1 - lam) * wb[perm]

def weighted_bce(logits, target, w):
    row = F.binary_cross_entropy_with_logits(logits, target, reduction="none").mean(dim=1)
    return (row * w).sum() / w.sum()

bce = nn.BCEWithLogitsLoss()

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro")

@torch.no_grad()
def evaluate():
    """a1-v2s-train's validation: first 5 s of each val clip, primary label only."""
    model.eval()
    preds, targets, loss_sum = [], [], 0.0
    for step, (xb, yb) in enumerate(dl_va):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits = model(xb)
        loss_sum += bce(logits.float(), yb).item() * len(xb)
        preds.append(torch.sigmoid(logits.float()).cpu().numpy()); targets.append(yb.cpu().numpy())
        if MAX_STEPS and step + 1 >= MAX_STEPS:
            break
    p, t = np.concatenate(preds), np.concatenate(targets)
    return loss_sum / len(p), macro_auc(t, p)

## Training
Guards: the projected epoch time is printed after 50 steps, and after epoch 1 the projected notebook total must be <= 1.9 h.

In [ ]:
def gpu_sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

history, best_auc, best_epoch, epoch_files = [], 0.0, None, {}
for epoch in range(1, EPOCHS + 1):
    files, k_s, k_f, batches, rng = make_plan(epoch)
    focal_sampler.batches = batches
    epoch_files[epoch] = dict(n=len(files), first=[FILE_NAMES[i] for i in files[:3]],
                              sha256=hashlib.sha256("\n".join(FILE_NAMES[i] for i in files).encode()).hexdigest())
    sound_dl = torch.utils.data.DataLoader(
        sound_lib.SoundFiles([SOUND_PATHS[i] for i in files], CFG["sr"], CFG["duration"]), batch_size=None,
        shuffle=False, num_workers=NS["sound_workers"], pin_memory=PIN,
        prefetch_factor=4 if NS["sound_workers"] > 0 else None)
    model.train(); pool.reset()
    t0 = time.time(); loss_sum, n_seen, wait_s, n_files, rows_f, rows_s = 0.0, 0, 0.0, 0, 0, 0
    f_it, s_it, s_left = iter(dl_tr), iter(sound_dl), True
    for step in range(STEPS):
        tw = time.time()
        xf = yf = None
        if k_f[step] > 0:
            xf, yf = next(f_it)
        wait_s += time.time() - tw
        need = int(k_s[step])
        while s_left and pool.count < max(POOL_TARGET, need):
            tw = time.time()
            try:
                wav, j = next(s_it)
            except StopIteration:
                s_left = False
                break
            wait_s += time.time() - tw
            rows = FILE_ROWS[files[int(j)]]
            if wav.shape != (len(rows), WIN_SAMPLES):
                raise RuntimeError(f"{FILE_NAMES[files[int(j)]]}: decoded {tuple(wav.shape)}, csv has {len(rows)} windows")
            with torch.no_grad():
                xs_new = mel_gpu(wav.to(DEVICE, non_blocking=True))          # float32, outside autocast
            pool.add(xs_new, PSEUDO_T[torch.from_numpy(rows).to(DEVICE)])
            n_files += 1
        xs, ys = pool.take(need, rng)
        xs_parts, ys_parts, ws_parts = [], [], []
        if xf is not None:
            xs_parts.append(xf.to(DEVICE, non_blocking=True)); ys_parts.append(yf.to(DEVICE, non_blocking=True))
            ws_parts.append(torch.full((len(xf),), NS["focal_weight"], device=DEVICE))
        if need > 0:
            xs_parts.append(xs[:, None]); ys_parts.append(ys)
            ws_parts.append(torch.full((need,), NS["sound_weight"], device=DEVICE))
        xb, yb, wb = torch.cat(xs_parts), torch.cat(ys_parts), torch.cat(ws_parts)
        assert len(xb) == BS, (len(xb), BS)
        rows_f += BS - need; rows_s += need
        if CFG["mixup"] and np.random.rand() < CFG["mixup_prob"]:
            xb, yb, wb = mixup(xb, yb, wb)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits = model(xb)
        loss = weighted_bce(logits.float(), yb, wb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(xb); n_seen += len(xb)
        if epoch == 1 and step + 1 == GUARD_STEPS:
            gpu_sync()
            per_step = (time.time() - t0) / GUARD_STEPS
            print(f"TIMING GUARD | after {GUARD_STEPS} steps: {per_step:.3f}s/step, data wait {wait_s:.1f}s | projected train part "
                  f"of an epoch {per_step * STEPS / 60:.1f} min, {EPOCHS} epochs {per_step * STEPS * EPOCHS / 3600:.2f} h", flush=True)
        if MAX_STEPS and step + 1 >= MAX_STEPS:
            break
    del f_it, s_it, sound_dl
    tr_loss = loss_sum / n_seen
    train_sec = time.time() - t0
    va_loss, va_auc = evaluate()
    epoch_sec = time.time() - t0
    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, val_auc=va_auc, train_sec=train_sec,
                        epoch_sec=epoch_sec, data_wait_sec=wait_s, focal_rows=rows_f, sound_rows=rows_s,
                        sound_files_decoded=n_files, steps=step + 1))
    flag = ""
    if va_auc > best_auc:          # as a1-v2s-train: model.pth = best clean val AUC
        best_auc, best_epoch = va_auc, epoch; torch.save(model.state_dict(), OUT / "model.pth"); flag = "  <- saved"
    pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
    print(f"epoch {epoch:2d} | train {tr_loss:.4f} | val {va_loss:.4f} | val AUC {va_auc:.4f} | train {train_sec:.0f}s | "
          f"epoch {epoch_sec:.0f}s | data wait {wait_s:.0f}s | rows focal {rows_f} sound {rows_s} | files {n_files}{flag}", flush=True)
    if epoch == 1:
        projected = time.time() - T_NOTEBOOK + (EPOCHS - 1) * epoch_sec + 120
        print(f"TIMING GUARD | after epoch 1: projected notebook total {projected / 3600:.2f} h (limit {BUDGET_H} h)", flush=True)
        if projected > BUDGET_H * 3600:
            raise RuntimeError(f"Projected {projected / 3600:.2f} h exceeds the {BUDGET_H} h limit, stopping")

torch.save(model.state_dict(), OUT / "model_final.pth")
if best_epoch is None:
    raise RuntimeError("no epoch improved on val AUC 0; model.pth was not written")
print(f"total epoch time {sum(h['epoch_sec'] for h in history) / 3600:.2f} h")
print(f"[ns-student-{TEACHER}] best val macro ROC-AUC: {best_auc:.4f} (epoch {best_epoch}) | final {history[-1]['val_auc']:.4f}")

## Save config and metadata
`cfg.json` has the keys that fast-infer reads (`model`, `norm`, `norm_top_db` and the mel settings).

In [ ]:
cfg_out = dict(CFG, best_val_auc=best_auc)
cfg_out["noise_db"] = list(CFG["noise_db"])
json.dump(cfg_out, open(OUT / "cfg.json", "w"), indent=1)
json.dump(species, open(OUT / "species.json", "w"))
pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
total = time.time() - T_NOTEBOOK
meta = dict(
    teacher=TEACHER, pseudo_file=PSEUDO_FILE, pseudo_kernel=PSEUDO_KERNEL, pseudo_kernel_version=PSEUDO_VERSION,
    sha256_pinned=PINNED_SHA256, sha256_found=SHA_FOUND, sha256_all_match=not bad,
    init=dict(backbone_ckpt=CFG["backbone_ckpt"], path=str(CKPT_PATH), load=LOAD_INFO, teacher_weights_used=False),
    sampling=dict(NS, windows_per_epoch=N_S, focal_train_clips=N_F, steps_per_epoch=STEPS, batch_size=BS,
                  pool_target_windows=POOL_TARGET, files_total=len(FILE_NAMES), epoch_files=epoch_files,
                  batch_composition="row types placed at random over the epoch, then cut into batches of 64; last partial batch dropped"),
    checkpoint_rule="model.pth = best clean val AUC (first 5 s, primary label) as in a1-v2s-train; model_final.pth = last epoch",
    best_epoch=best_epoch, best_val_auc=best_auc, final_val_auc=history[-1]["val_auc"], epochs=EPOCHS,
    mel=dict(transform="TorchA1Mel (ns-mel-check)", pad_mode=LIBROSA_PAD_MODE, device=DEVICE),
    amp=DEVICE == "cuda", versions=VERSIONS, device=DEVICE_NAME, smoke=SMOKE, seconds=dict(notebook=total),
)
json.dump(meta, open(OUT / "student_meta.json", "w"), indent=1, default=str)
print(f"TIMING | notebook {total / 60:.1f} min | epochs {sum(h['epoch_sec'] for h in history) / 60:.1f} min | "
      f"data wait {sum(h['data_wait_sec'] for h in history) / 60:.1f} min | device {DEVICE_NAME}")